# End-to-End Memory Rule Demo

这份 notebook 走完整条主线，但只使用隔离的 demo 数据，不污染正式库。

它会演示四件事：

1. 把几道错题写入临时事件库
2. 从事件库重建 `student_memory_profile`
3. 对比 `coach_agent` 在“无画像 / 有画像”时的本地规则变化
4. 对比 `review_scheduler` 在“无画像 / 有画像”时的排序变化

默认不依赖真实模型调用，所以跑起来稳定、可复现。
最后会留一个可选单元，供你自己决定要不要真实调用 `coach_agent`。

In [ ]:
import difflib
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from coach_agent import (
    CoachSession,
    ErrorType,
    FoundryCoachAgent,
    analyze_student_reply,
    apply_student_memory_bias,
    build_reply_analysis_fallback,
    build_student_memory_context,
    build_student_memory_strategy_note,
    build_turn_prompt,
    get_coach_strategy,
    infer_completed_from_reply,
)
from review_scheduler import build_review_schedule, load_review_state
from student_memory_events import (
    binding_result_to_memory_event,
    coach_response_to_memory_event,
    diagnosis_result_to_memory_event,
    review_state_update_to_memory_event,
)
from student_memory_store import append_event, build_profile_from_store, load_events, summarize_store

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'end_to_end_memory_rule_demo'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
DEMO_EVENTS_PATH = SCRATCH_DIR / 'student_memory_events_demo.jsonl'

def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2))

def show_block(title: str, text: str) -> None:
    print(f'\n{title}')
    print('-' * len(title))
    print(text.strip() if str(text).strip() else '（空）')

def show_prompt_diff(left: str, right: str, *, left_name: str, right_name: str) -> None:
    diff = difflib.unified_diff(
        left.splitlines(),
        right.splitlines(),
        fromfile=left_name,
        tofile=right_name,
        lineterm='',
    )
    diff_text = '\n'.join(diff).strip()
    print(diff_text if diff_text else '两份 prompt 完全一致。')

def reset_demo_store(path: Path):
    if path.exists():
        path.unlink()

def question_binding(primary_node_id, secondary_node_ids):
    return {
        'primary_node_id': primary_node_id,
        'secondary_node_ids': secondary_node_ids,
        'linked_node_ids': [primary_node_id, *secondary_node_ids],
    }


## 1. 读取三道例题

这里直接使用现成的 `wrong_binder` fixtures。
我们故意挑三道风格不一样的题，方便长期画像更快形成偏向。

In [ ]:
FIXTURE_DIR = PROJECT_ROOT / 'harness' / 'fixtures' / 'wrong_binder'

CASE_PATHS = [
    FIXTURE_DIR / 'binder_case_sequence_shift.json',
    FIXTURE_DIR / 'binder_case_vector_angle.json',
    FIXTURE_DIR / 'binder_case_circle_distance.json',
]

CASES = [json.loads(path.read_text(encoding='utf-8')) for path in CASE_PATHS]

STUDENT_ID = 'demo_e2e_memory_rule'
SESSION_ID = 'session_e2e_memory_rule_001'
BASE_TIME = datetime(2026, 6, 24, 20, 0, tzinfo=timezone(timedelta(hours=8)))

CASE_SUMMARY = []
for item in CASES:
    question = item['question_payload']
    CASE_SUMMARY.append(
        {
            'case_id': item['case_id'],
            'description': item['description'],
            'question_type': question['question_type'],
            'stem': question['stem'],
            'student_answer': question['student_answer'],
            'source_name': question['source_name'],
            'source_section': question['source_section'],
        }
    )

show_json(CASE_SUMMARY)


## 2. 设定绑定、诊断、引导、复习结果

这里不走真实模型，而是手工构造一条结构真实的事件链。
目的不是测模型，而是测“入库 -> 画像 -> 规则 -> 下游变化”这条主线。

In [ ]:
QUESTION_SCRIPTS = [
    {
        'question_id': 'wq_demo_seq_001',
        'case': CASES[0],
        'binding': question_binding(
            'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
            ['math.数列与不等式.数列.基础概念.数列递推公式解读'],
        ),
        'diagnosis': {
            'error_type': 'missing_strategy',
            'confidence': 0.84,
            'reason': '学生知道从递推式入手，但不会主动提出构造新数列这个中间目标。',
            'evidence': '我知道要从递推式入手，但不知道为什么要构造 a_n+1/2。',
            'coach_mode': 'socratic_light',
            'coach_trap': '学生知道局部知识，但不会先确定中间目标。',
            'coach_prompt': '先给一个很小的起点，只透露下一步，不要直接给完整答案。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '我们先别急着算结果，先想这道题要把递推式改造成什么结构。你试着说说，新数列最想变成哪种前后项关系？',
            'reply_quality': 'empty',
            'strategy': {
                'mode': 'socratic_light',
                'trap': '学生知道局部知识，但不会先确定中间目标。',
                'prompt': '先让学生明确中间目标，再推进具体变形。',
            },
            'turn_index': 1,
            'done': False,
            'stop_reason': 'continue',
            'reply_analysis': {
                'quality': 'empty',
                'understands': False,
                'completed': False,
                'reason': '学生还说不出要把递推式改造成固定比的结构。',
            },
        },
        'review_result': 'wrong',
    },
    {
        'question_id': 'wq_demo_vec_001',
        'case': CASES[1],
        'binding': question_binding(
            'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
            ['math.三角函数_平面向量与解三角形.平面向量.数量积几何意义'],
        ),
        'diagnosis': {
            'error_type': 'missing_strategy',
            'confidence': 0.78,
            'reason': '学生知道数量积，但不会把向量和关系先转成模平方，再落到夹角公式。',
            'evidence': '我能想到数量积，但不会把向量和的模平方展开到夹角公式。',
            'coach_mode': 'socratic_light',
            'coach_trap': '学生会局部公式，但不会先做结构转换。',
            'coach_prompt': '先提醒把向量和关系改写成模平方，再问下一步。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '这题不是直接套夹角公式，而是先把 a+b=-c 变成长度关系。你先试着说说，为什么这里先考虑 |a+b|^2 会更方便？',
            'reply_quality': 'weak',
            'strategy': {
                'mode': 'socratic_light',
                'trap': '学生会局部公式，但不会先做结构转换。',
                'prompt': '先点出结构转换，再只追问一个关键中间量。',
            },
            'turn_index': 1,
            'done': False,
            'stop_reason': 'continue',
            'reply_analysis': {
                'quality': 'weak',
                'understands': True,
                'completed': False,
                'reason': '学生已经能想到数量积，但还不会主动做模平方转换。',
            },
        },
        'review_result': 'partial',
    },
    {
        'question_id': 'wq_demo_circle_001',
        'case': CASES[2],
        'binding': question_binding(
            'math.解析几何.直线的方程.点到直线的距离公式',
            ['math.解析几何.圆的方程.圆的标准方程'],
        ),
        'diagnosis': {
            'error_type': 'concept_gap',
            'confidence': 0.81,
            'reason': '学生不会把张角存在性先转成圆心到直线距离范围，说明对几何判定条件理解不稳。',
            'evidence': '我知道要用点到直线距离，但不知道为什么先转成圆心到直线距离不超过 2。',
            'coach_mode': 'direct_explain',
            'coach_trap': '学生缺少必要几何判定概念，继续追问会放大挫败。',
            'coach_prompt': '先讲清“存在张角60°”为什么等价于圆心距离约束，再回到公式。',
            'source': 'manual_demo',
        },
        'coach_response': {
            'content': '这题关键不是先算 t，而是先判断什么情况下直线上能找到这样的点 P。你先记住一个核心转化：张角存在性，先变成圆心到直线的距离范围。',
            'reply_quality': 'good',
            'strategy': {
                'mode': 'direct_explain',
                'trap': '学生缺少必要几何判定概念，继续追问会放大挫败。',
                'prompt': '先补清概念判定，再回到距离公式。',
            },
            'turn_index': 1,
            'done': True,
            'stop_reason': 'student_understood',
            'reply_analysis': {
                'quality': 'good',
                'understands': True,
                'completed': True,
                'reason': '学生已经能接受先转成圆心到直线距离范围的思路。',
            },
        },
        'review_result': 'correct',
    },
]

show_json(
    [
        {
            'question_id': item['question_id'],
            'primary_node_id': item['binding']['primary_node_id'],
            'secondary_node_ids': item['binding']['secondary_node_ids'],
            'diagnosis_error_type': item['diagnosis']['error_type'],
            'review_result': item['review_result'],
        }
        for item in QUESTION_SCRIPTS
    ]
)


## 3. 把事件写进隔离的 demo store

这里会写四类事件：

- `binding`
- `diagnosis`
- `coach`
- `review`


In [ ]:
reset_demo_store(DEMO_EVENTS_PATH)

WRITTEN_EVENTS = []

for index, item in enumerate(QUESTION_SCRIPTS):
    occurred_base = BASE_TIME + timedelta(minutes=index * 25)
    question = item['case']['question_payload']

    binding_event = binding_result_to_memory_event(
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        primary_node_id=item['binding']['primary_node_id'],
        secondary_node_ids=item['binding']['secondary_node_ids'],
        candidate_node_ids=item['binding']['linked_node_ids'],
        binding_source='student_confirmed',
        occurred_at=occurred_base.isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
        binding_confidence=0.9,
    )
    append_event(binding_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(binding_event)

    diagnosis_event = diagnosis_result_to_memory_event(
        item['diagnosis'],
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        binding=item['binding'],
        occurred_at=(occurred_base + timedelta(minutes=2)).isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(diagnosis_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(diagnosis_event)

    coach_event = coach_response_to_memory_event(
        item['coach_response'],
        question_id=item['question_id'],
        question_type=question.get('question_type'),
        binding=item['binding'],
        occurred_at=(occurred_base + timedelta(minutes=5)).isoformat(),
        error_type=item['diagnosis']['error_type'],
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(coach_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(coach_event)

    review_event = review_state_update_to_memory_event(
        {
            'action': 'review_result',
            'target_type': 'wrong_question',
            'target_id': item['question_id'],
            'result': item['review_result'],
            'student_id': STUDENT_ID,
            'session_id': SESSION_ID,
            'executed_at': (occurred_base + timedelta(days=2)).isoformat(),
            'updated_payload': {
                'example_question_states': [
                    {
                        'question_id': item['question_id'],
                        'question_type': question.get('question_type'),
                        'linked_node_ids': item['binding']['linked_node_ids'],
                        'primary_node_ids': [item['binding']['primary_node_id']],
                        'secondary_node_ids': item['binding']['secondary_node_ids'],
                        'last_result': item['review_result'],
                        'review_count': 1,
                    }
                ],
                'knowledge_point_states': [
                    {
                        'node_id': item['binding']['primary_node_id'],
                        'linked_question_ids': [item['question_id']],
                        'mastery': 0.25 if item['review_result'] == 'wrong' else 0.55 if item['review_result'] == 'partial' else 0.82,
                        'stability': 0.4 if item['review_result'] == 'wrong' else 1.2 if item['review_result'] == 'partial' else 2.8,
                    }
                ],
            },
        }
    )
    append_event(review_event, path=DEMO_EVENTS_PATH)
    WRITTEN_EVENTS.append(review_event)

print('demo store path:', DEMO_EVENTS_PATH)
print('written events:', len(WRITTEN_EVENTS))


## 4. 看事件库里到底写了什么

In [ ]:
show_json(summarize_store(DEMO_EVENTS_PATH).as_dict())
show_json(load_events(path=DEMO_EVENTS_PATH, student_id=STUDENT_ID))


## 5. 从事件库重建 `student_memory_profile`

In [ ]:
PROFILE = build_profile_from_store(
    student_id=STUDENT_ID,
    path=DEMO_EVENTS_PATH,
)

show_json(PROFILE)


## 6. 单独看长期画像摘要

这部分就是后面要喂给 `coach_agent` 和 `review_scheduler` 的核心摘要。

In [ ]:
show_json(PROFILE['personalization_summary'])


## 7. 看 `coach_agent` 在“无画像 / 有画像”时的变化

这里先做本地可解释对比，不调用真实模型。
我们固定用第一道题来比较。

In [ ]:
TARGET = QUESTION_SCRIPTS[0]
QUESTION = TARGET['case']['question_payload']
PROBLEM_TEXT = QUESTION['stem']
STUDENT_REPLY = QUESTION['student_answer']
STUDENT_PROFILE = '学生能从递推式入手，但经常不会自己提出构造新数列这种中间目标。'
CURRENT_ERROR_TYPE = ErrorType.MISSING_STRATEGY

reply_analysis = build_reply_analysis_fallback(
    analyze_student_reply(STUDENT_REPLY),
    '本 notebook 为了稳定对比，不调用 reply analyzer，直接使用本地 fallback。',
    completed=infer_completed_from_reply(STUDENT_REPLY),
)

base_strategy = get_coach_strategy(
    CURRENT_ERROR_TYPE,
    reply_analysis.quality,
    turn_index=0,
    total_turns=4,
    understands=reply_analysis.understands,
    completed=reply_analysis.completed,
)

session_plain = CoachSession(
    problem_text=PROBLEM_TEXT,
    error_type=CURRENT_ERROR_TYPE,
    student_profile=STUDENT_PROFILE,
    initial_student_reply=STUDENT_REPLY,
)

session_memory = CoachSession(
    problem_text=PROBLEM_TEXT,
    error_type=CURRENT_ERROR_TYPE,
    student_profile=STUDENT_PROFILE,
    student_memory_profile=PROFILE,
    student_memory_context=build_student_memory_context(PROFILE),
    initial_student_reply=STUDENT_REPLY,
)

strategy_plain = apply_student_memory_bias(base_strategy, session_plain)
strategy_memory = apply_student_memory_bias(base_strategy, session_memory)

prompt_plain = build_turn_prompt(
    session=session_plain,
    student_reply=STUDENT_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_plain,
)

prompt_memory = build_turn_prompt(
    session=session_memory,
    student_reply=STUDENT_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_memory,
)

show_json(
    {
        'reply_analysis': reply_analysis.as_dict(),
        'base_strategy': base_strategy.as_dict(),
        'strategy_without_memory': strategy_plain.as_dict(),
        'strategy_with_memory': strategy_memory.as_dict(),
    }
)


In [ ]:
show_block('一、无长期画像：student_memory_context', session_plain.student_memory_context)
show_block('二、有长期画像：student_memory_context', session_memory.student_memory_context)
show_block('三、无长期画像：本轮策略话术', strategy_plain.as_prompt_block())
show_block('四、有长期画像：本轮策略话术', strategy_memory.as_prompt_block())
show_block('五、长期画像额外加上的策略提示', build_student_memory_strategy_note(base_strategy, session_memory))


In [ ]:
show_block('六、无长期画像：turn prompt', prompt_plain)
show_block('七、有长期画像：turn prompt', prompt_memory)
show_block('八、turn prompt diff', '')
show_prompt_diff(prompt_plain, prompt_memory, left_name='without_memory', right_name='with_memory')


## 8. 可选：真实调用 `coach_agent`

默认注释掉。
如果你本机环境和 deployment 都好了，可以打开运行，看真正回复文本的差异。

In [ ]:
# USE_REAL_COACH = False
# if USE_REAL_COACH:
#     coach = FoundryCoachAgent(use_ai_reply_analyzer=False)
#     response_plain = coach.start(session_plain, student_reply=STUDENT_REPLY)
#     response_memory = coach.start(session_memory, student_reply=STUDENT_REPLY)
#     show_block('九、无长期画像：首轮 coach 回复', response_plain.content)
#     show_block('十、有长期画像：首轮 coach 回复', response_memory.content)
#     show_json(
#         {
#             'without_memory_strategy': response_plain.strategy.as_dict(),
#             'with_memory_strategy': response_memory.strategy.as_dict(),
#             'without_memory_reply_analysis': response_plain.reply_analysis.as_dict(),
#             'with_memory_reply_analysis': response_memory.reply_analysis.as_dict(),
#         }
#     )


## 9. 看 `review_scheduler` 在“无画像 / 有画像”时的变化

这里不再读取外部 review seed，而是直接用同一批 `wq_demo_*` 题目构造一个对齐的 review state。
这样题目侧和知识点侧都能真正吃到长期画像。

In [ ]:
REVIEW_STATE = {
    'knowledge_point_states': [
        {
            'node_id': 'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
            'state': 'learning',
            'mastery': 0.22,
            'stability': 0.35,
            'linked_question_ids': ['wq_demo_seq_001'],
            'priority_note': '递推转化核心叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=5)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=3)).isoformat(),
            'next_review_at': (BASE_TIME - timedelta(hours=10)).isoformat(),
        },
        {
            'node_id': 'math.数列与不等式.数列.基础概念.数列递推公式解读',
            'state': 'learning',
            'mastery': 0.2,
            'stability': 0.3,
            'linked_question_ids': ['wq_demo_seq_001'],
            'priority_note': '递推阅读叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=5)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=4)).isoformat(),
            'next_review_at': (BASE_TIME - timedelta(hours=8)).isoformat(),
        },
        {
            'node_id': 'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
            'state': 'review',
            'mastery': 0.5,
            'stability': 1.1,
            'linked_question_ids': ['wq_demo_vec_001'],
            'priority_note': '向量夹角核心叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=6)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=2)).isoformat(),
            'next_review_at': (BASE_TIME - timedelta(hours=2)).isoformat(),
        },
        {
            'node_id': 'math.三角函数_平面向量与解三角形.平面向量.数量积几何意义',
            'state': 'review',
            'mastery': 0.52,
            'stability': 1.0,
            'linked_question_ids': ['wq_demo_vec_001'],
            'priority_note': '数量积支持叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=6)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=2)).isoformat(),
            'next_review_at': (BASE_TIME + timedelta(hours=6)).isoformat(),
        },
        {
            'node_id': 'math.解析几何.直线的方程.点到直线的距离公式',
            'state': 'review',
            'mastery': 0.62,
            'stability': 1.8,
            'linked_question_ids': ['wq_demo_circle_001'],
            'priority_note': '解析几何距离叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=7)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=1)).isoformat(),
            'next_review_at': (BASE_TIME + timedelta(hours=10)).isoformat(),
        },
        {
            'node_id': 'math.解析几何.圆的方程.圆的标准方程',
            'state': 'review',
            'mastery': 0.58,
            'stability': 1.5,
            'linked_question_ids': ['wq_demo_circle_001'],
            'priority_note': '圆标准方程叶子',
            'first_seen_at': (BASE_TIME - timedelta(days=7)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=1)).isoformat(),
            'next_review_at': (BASE_TIME + timedelta(hours=4)).isoformat(),
        },
    ],
    'example_question_states': [
        {
            'question_id': 'wq_demo_seq_001',
            'question_type': '证明题',
            'state': 'learning',
            'mastery': 0.18,
            'stability': 0.3,
            'linked_node_ids': [
                'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
                'math.数列与不等式.数列.基础概念.数列递推公式解读',
            ],
            'primary_node_ids': ['math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化'],
            'secondary_node_ids': ['math.数列与不等式.数列.基础概念.数列递推公式解读'],
            'last_result': 'wrong',
            'review_count': 2,
            'priority_note': '数列递推示例题',
            'first_seen_at': (BASE_TIME - timedelta(days=5)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=2)).isoformat(),
            'next_review_at': (BASE_TIME - timedelta(hours=6)).isoformat(),
        },
        {
            'question_id': 'wq_demo_vec_001',
            'question_type': '选择题',
            'state': 'review',
            'mastery': 0.48,
            'stability': 1.0,
            'linked_node_ids': [
                'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
                'math.三角函数_平面向量与解三角形.平面向量.数量积几何意义',
            ],
            'primary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.向量夹角公式'],
            'secondary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.数量积几何意义'],
            'last_result': 'partial',
            'review_count': 2,
            'priority_note': '向量夹角示例题',
            'first_seen_at': (BASE_TIME - timedelta(days=6)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=2)).isoformat(),
            'next_review_at': (BASE_TIME - timedelta(hours=1)).isoformat(),
        },
        {
            'question_id': 'wq_demo_circle_001',
            'question_type': '选择题',
            'state': 'review',
            'mastery': 0.66,
            'stability': 1.8,
            'linked_node_ids': [
                'math.解析几何.直线的方程.点到直线的距离公式',
                'math.解析几何.圆的方程.圆的标准方程',
            ],
            'primary_node_ids': ['math.解析几何.直线的方程.点到直线的距离公式'],
            'secondary_node_ids': ['math.解析几何.圆的方程.圆的标准方程'],
            'last_result': 'correct',
            'review_count': 2,
            'priority_note': '圆与直线示例题',
            'first_seen_at': (BASE_TIME - timedelta(days=7)).isoformat(),
            'last_reviewed_at': (BASE_TIME - timedelta(days=1)).isoformat(),
            'next_review_at': (BASE_TIME + timedelta(hours=8)).isoformat(),
        },
    ],
}

show_json(REVIEW_STATE)

schedule_plain = build_review_schedule(
    REVIEW_STATE,
    now=BASE_TIME,
    student_memory_profile=None,
)

schedule_memory = build_review_schedule(
    REVIEW_STATE,
    now=BASE_TIME,
    student_memory_profile=PROFILE,
)

show_json(
    {
        'without_memory_review_plan': schedule_plain.review_plan,
        'with_memory_review_plan': schedule_memory.review_plan,
    }
)


In [ ]:
plain_top = [
    {
        'node_id': item['node_id'],
        'priority_score': item['priority_score'],
        'memory_priority_boost': item['memory_priority_boost'],
        'memory_priority_reason': item['memory_priority_reason'],
    }
    for item in schedule_plain.node_priorities[:5]
]

memory_top = [
    {
        'node_id': item['node_id'],
        'priority_score': item['priority_score'],
        'memory_priority_boost': item['memory_priority_boost'],
        'memory_priority_reason': item['memory_priority_reason'],
    }
    for item in schedule_memory.node_priorities[:5]
]

show_json({'top_nodes_without_memory': plain_top, 'top_nodes_with_memory': memory_top})


In [ ]:
plain_questions = [
    {
        'question_id': item['question_id'],
        'priority_score': item['priority_score'],
        'memory_priority_boost': item['memory_priority_boost'],
        'memory_priority_reason': item['memory_priority_reason'],
    }
    for item in schedule_plain.question_priorities[:5]
]

memory_questions = [
    {
        'question_id': item['question_id'],
        'priority_score': item['priority_score'],
        'memory_priority_boost': item['memory_priority_boost'],
        'memory_priority_reason': item['memory_priority_reason'],
    }
    for item in schedule_memory.question_priorities[:5]
]

show_json({'top_questions_without_memory': plain_questions, 'top_questions_with_memory': memory_questions})


## 10. 最后总结

你现在应该能直接看到三层变化：

1. 错题如何被写进临时事件库
2. 事件如何长成 `student_memory_profile`
3. 画像如何轻度影响 `coach_agent` 与 `review_scheduler`

而且整条链路都只用了 `scratch/end_to_end_memory_rule_demo`，不会污染正式数据。